# LoveDA training on Colab

This notebook runs the whole pipeline on a free GPU. It gets the code, installs the extra dependencies, downloads LoveDA, trains DeepLabV3 ResNet50, then evaluates, analyzes and shows the results.

Before running, switch the runtime to a GPU. Use the menu Runtime then Change runtime type then pick a GPU such as T4.

Colab sessions are temporary, so the last cell copies your run to Google Drive. Do that or you lose the checkpoint when the session ends.

## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())
!nvidia-smi -L

## 2. Get the code

Two ways. Clone from GitHub if you pushed the project, or upload a zip of the project folder. Run only one of the two cells below.

In [ ]:
# Option A. Clone from GitHub.
REPO_URL = "https://github.com/ShanmukhUpad/semantic-segmentation-feature.git"
!git clone $REPO_URL
%cd semantic-segmentation-feature

In [ ]:
# Option B. Upload a zip of the project instead of cloning, then unzip it.
# from google.colab import files
# uploaded = files.upload()            # choose your project zip
# import zipfile, io
# name = next(iter(uploaded))
# with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
#     archive.extractall(".")
# %cd semantic-segmentation-feature

## 3. Install the extra dependencies

Colab already provides a CUDA build of torch and torchvision, so do not reinstall those. Installing the pinned CPU torch from requirements would replace the GPU build and slow everything down, so only the lighter extras are installed here.

In [ ]:
!pip -q install seaborn tensorboard pyyaml tqdm scikit-learn gradio

## 4. Download LoveDA

This pulls Train and Val from the official Zenodo record and arranges them. It is a few gigabytes, so it takes a few minutes.

In [ ]:
!python scripts/download_loveda.py --root data/raw/LoveDA

## 5. Train on the GPU

This uses the LoveDA config with the two T4 speed knobs turned on, namely mixed precision and image size 384. With them a T4 epoch takes about 8 minutes, so 15 epochs finish in roughly 2 to 2.5 hours and reach about 0.50 mean IoU on the validation split. Training at the default 512 with mixed precision off is much slower, around 22 minutes per epoch. Raise train.epochs or drop the image size override for a stronger full resolution run when you have the GPU time.

In [ ]:
!python scripts/train.py --config configs/default.yaml --set train.epochs=15 dataloader.batch_size=8 dataloader.num_workers=2 train.amp=true dataset.image_size=[384,384]

## 6. Evaluate and analyze on the validation split

LoveDA Test labels are withheld, so the validation split is the held out set.

In [ ]:
!python scripts/evaluate.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val
!python scripts/analyze.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val

## 7. View the results inline

In [ ]:
import glob
from IPython.display import Image as IPImage, Markdown, display
base = "results/loveda_deeplabv3/analysis"
display(Markdown(open(base + "/report.md").read()))
for path in sorted(glob.glob(base + "/*.png")) + sorted(glob.glob(base + "/error_maps/*.png")):
    print(path)
    display(IPImage(path))

## 8. Label free failure detection anywhere in the world

The model can score aerial imagery from any place on earth even though no labels exist there. First fit_ood.py builds a reference of what the LoveDA training tiles look like in backbone feature space. Then scan_failures.py takes any folder of images, slides a window over each one at native resolution so the 0.3 m ground sample distance is preserved, and writes failure heatmaps plus a ranking of the least trustworthy images. High entropy or margin risk marks ambiguous pixels. A high novelty score marks scenes unlike the training data, where even confident predictions deserve doubt.

The demo below scans a few LoveDA validation images, which are in domain, so expect low novelty. To scan your own imagery from anywhere, upload aerial tiles near 0.3 m per pixel into data/world_samples. No label is needed.

In [ ]:
!python scripts/fit_ood.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --num-images 300

In [ ]:
import glob, os, shutil
os.makedirs("data/world_samples", exist_ok=True)
for path in sorted(glob.glob("data/raw/LoveDA/Val/Urban/images_png/*.png"))[:6]:
    shutil.copy(path, "data/world_samples")
# Drop your own aerial tiles into data/world_samples to scan any place on earth.

!python scripts/scan_failures.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --images data/world_samples --output-dir results/loveda_deeplabv3/scan

import pandas as pd
from IPython.display import Image as IPImage, display
display(pd.read_csv("results/loveda_deeplabv3/scan/rankings.csv"))
for path in sorted(glob.glob("results/loveda_deeplabv3/scan/panels/*.png"))[:3]:
    display(IPImage(path))

## 9. Validate the failure signals against real labels

The failure maps above are only worth trusting if they truly predict errors. This check runs the signals on the labeled validation split and measures how well each one ranks wrong pixels above correct ones. AUROC of 0.5 is chance and 1.0 is perfect. It also reports how well the per image mean signal tracks the true per image error rate.

This first cell runs in domain on LoveDA validation, which mostly proves the plumbing. The cross domain cell after it is the test that matters.

In [ ]:
!python scripts/validate_signals.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --limit 300

import json
from IPython.display import Image as IPImage, display
base = "results/loveda_deeplabv3/validation/loveda_val"
print(json.dumps(json.load(open(base + "/metrics.json"))["pixel"], indent=2))
for name in ["risk_coverage.png", "failure_deciles.png", "failure_vs_error.png"]:
    display(IPImage(base + "/" + name))

### Cross domain test on ISPRS Potsdam

This is the real proof. The model trained on Chinese imagery while Potsdam is German urban imagery at 0.05 m per pixel, so the geography, the sensor and the scene style all change at once. The cell below pulls the ISPRS benchmark from a public Kaggle mirror, cuts the 38 orthophotos into 2304 pixel patches so the resized network input lands back at 0.3 m per pixel, then reruns the validation with the Potsdam class mapping and the LoveDA run above as the novelty baseline. Run the LoveDA cell first since its per_image.csv is the baseline here. If AUROC and the correlations stay clearly above chance, the label free signals survive domain shift.

In [ ]:
import kagglehub
src = kagglehub.dataset_download("aletbm/urban-segmentation-isprs")
print("dataset at", src)

!python scripts/prepare_potsdam.py --images "{src}/Potsdam/Images" --labels "{src}/Potsdam/Labels" --output data/raw/PotsdamPatches

!python scripts/validate_signals.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --set dataset.name=potsdam dataset.root=data/raw/PotsdamPatches --label-map potsdam_to_loveda --baseline-csv results/loveda_deeplabv3/validation/loveda_val/per_image.csv

import json
from IPython.display import Image as IPImage, display
base = "results/loveda_deeplabv3/validation/potsdam_val"
print(json.dumps(json.load(open(base + "/metrics.json"))["pixel"], indent=2))
for name in ["risk_coverage.png", "failure_deciles.png", "failure_vs_error.png", "novelty_hist.png"]:
    display(IPImage(base + "/" + name))

## 10. Training curves and sample images in TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/loveda_deeplabv3/tensorboard

## 11. Interactive prediction app

Launch the upload GUI with a public link. Open the printed gradio.live URL, upload an aerial tile and see the real prediction. Without a ground truth upload the app shows the label free failure maps and the novelty verdict. Stop the cell to shut the app down.

In [ ]:
!python scripts/app.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --share

## 12. Save the run to Google Drive

Colab sessions are temporary. Run this to keep the checkpoint, report and figures.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r results/loveda_deeplabv3 "/content/drive/MyDrive/loveda_deeplabv3"
print("copied run to Google Drive")